In [2]:
from sklearn.datasets import load_iris
import pandas as pd
import numpy as np

data = load_iris()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

In [5]:
def calcular_limites_iqr(X):
    Q1 = X.quantile(0.25)
    Q3 = X.quantile(0.75)
    IQR = Q3 - Q1
    return (Q1 - 1.5 * IQR), (Q3 + 1.5 * IQR)

def calcular_limites_quantil(X, lim_inf=0.05, lim_sup=0.95):
    return X.quantile(lim_inf), X.quantile(lim_sup)

def remover_outliers_iqr(X, y):
    df = pd.DataFrame(X)
    lim_inf, lim_sup = calcular_limites_iqr(df)
    mascara = ~((df < lim_inf) | (df > lim_sup)).any(axis=1)
    return X[mascara], y[mascara]

def remover_outliers_quantil(X, y):
    df = pd.DataFrame(X)
    lim_inf, lim_sup = calcular_limites_quantil(df)
    mascara = ~((df < lim_inf) | (df > lim_sup)).any(axis=1)
    return X[mascara], y[mascara]

def substituir_outliers_iqr(X, y):
    df = pd.DataFrame(X)
    lim_inf, lim_sup = calcular_limites_iqr(df)
    df = df.clip(lower=lim_inf, upper=lim_sup, axis=1)
    return df.values, y

def substituir_outliers_quantil(X, y):
    df = pd.DataFrame(X)
    lim_inf, lim_sup = calcular_limites_quantil(df)
    df = df.clip(lower=lim_inf, upper=lim_sup, axis=1)
    return df.values, y

from sklearn.preprocessing import PowerTransformer, QuantileTransformer, MinMaxScaler, StandardScaler, RobustScaler, KBinsDiscretizer

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from imblearn import FunctionSampler

pre_processadores = {
    'Baseline': 'passthrough',
    
    'Remoção IQR': FunctionSampler(func=remover_outliers_iqr),
    'Remoção Quantis': FunctionSampler(func=remover_outliers_quantil),
    'Substituição IQR': FunctionSampler(func=substituir_outliers_iqr),
    'Substituição Quantis': FunctionSampler(func=substituir_outliers_quantil),
    
    'Transformação Yeo-Johnson': PowerTransformer(method='yeo-johnson'),
    'Transformação Quantil': QuantileTransformer(output_distribution='normal'),
    
    'Normalização Min-Max': MinMaxScaler(),
    'Padronização': StandardScaler(),
    'Escalonamento Robusto': RobustScaler(),
    
    'Discretização por Largura': KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='uniform'),
    'Discretização por Frequência': KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile'),
    
    'Subamostragem Aleatória': RandomUnderSampler(random_state=42),
    'Sobreamostragem (SMOTE)': SMOTE(random_state=42)
}

from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

algoritmos = {
    'Naive Bayes Gaussiano': GaussianNB(),
    'Regressão Logística': LogisticRegression(),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(),
    'Floresta Aleatória': RandomForestClassifier(),
    'MLP': MLPClassifier()
}

from sklearn.model_selection import StratifiedKFold, cross_validate
from imblearn.pipeline import Pipeline
from tqdm import tqdm

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
resultados = []

total_combinacoes = len(algoritmos) * len(pre_processadores)

pbar = tqdm(total=total_combinacoes)

for nome_alg, alg in algoritmos.items():
    for nome_prep, prep in pre_processadores.items():
        
        pbar.set_postfix(alg=nome_alg, prep=nome_prep)
        
        pipeline = Pipeline(steps=[('prep', prep), ('alg', alg)])
            
        scores = cross_validate(
            pipeline, X, y, 
            cv=cv, 
            scoring='f1_weighted', 
            n_jobs=-1
        )
        
        resultados.append({
            'Algoritmo': nome_alg,
            'Pre_Processamento': nome_prep,
            'F1_Medio': np.mean(scores['test_score']),
            'Tempo_Treino_Medio': np.mean(scores['fit_time'])
        })
        
        pbar.update(1)

pbar.close()

df_final = pd.DataFrame(resultados)

df_final

100%|██████████| 84/84 [00:06<00:00, 12.91it/s, alg=MLP, prep=Sobreamostragem (SMOTE)]                       


,Algoritmo,Pre_Processamento,F1_Medio,Tempo_Treino_Medio
0,Naive Bayes Gaussiano,Baseline,0.952576,0.002770
1,Naive Bayes Gaussiano,Remoção IQR,0.952576,0.007428
2,Naive Bayes Gaussiano,Remoção Quantis,0.945842,0.006493
3,Naive Bayes Gaussiano,Substituição IQR,0.952576,0.007233
4,Naive Bayes Gaussiano,Substituição Quantis,0.945842,0.007519
...,...,...,...,...
79,MLP,Escalonamento Robusto,0.940202,0.108651
80,MLP,Discretização por Largura,0.939916,0.104487
81,MLP,Discretização por Frequência,0.930186,0.106320
82,MLP,Subamostragem Aleatória,0.972643,0.105365


#### Dúvidas:
- Como trataremos os nulos?
- Como trataremos os categóricos?
- Testaremos mais de um valor para tratamentos de outliers com quantis (lim_inf, lim_sup)?
- Testaremos mais de um valor para discretização (n_bins e encoder)?
- Adicionaremos seleção de variáveis?
- Adicionaremos PCA?
- Testaremos mais de um tipo de cv?
- Adicionaremos mais alguma métrica?
- Adicionaremos alguma outra métrica de performace (como uso de memória)?
- Adicionaremos otimização de hiperparâmtros?